# LAWLESS — P0 Physics Foundation

**Evolution of adaptive control under non-stationary physics**

This notebook clones the GitHub repository, checks the runtime, installs the project, runs the P0 test suite, executes a deterministic sanity simulation, and benchmarks CPU/GPU physics stepping. P0 deliberately contains no learning agents or evolutionary optimization yet.

In [ ]:
#@title 1. Clone LAWLESS
REPO_URL = "https://github.com/manav4u/LAWLESS.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

from pathlib import Path
import os, subprocess, sys

repo_dir = Path('/content/LAWLESS')
if repo_dir.exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository:', Path.cwd())
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
#@title 2. Inspect runtime
import platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('Using CPU; P0 remains runnable without a GPU.')

In [ ]:
#@title 3. Install LAWLESS and test dependencies
%pip install -q -e '.[dev]'
print('Installation complete')

In [ ]:
#@title 4. Run P0 tests
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], text=True)
if result.returncode != 0:
    raise RuntimeError('P0 tests failed; inspect test output before proceeding.')
print('P0 test suite passed.')

In [ ]:
#@title 5. Deterministic physics sanity check
import torch
from lawless.physics import PhysicsConfig, make_state, step

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = PhysicsConfig()
state = make_state((4, 256), device=device, dtype=torch.float32, energy=cfg.initial_energy)
actions = torch.zeros((4, 256, 2), device=device)
actions[..., 0] = 0.35
gravity = torch.tensor([0.0, 4.0], device=device)

for _ in range(round(2.0 / cfg.dt)):
    state, diagnostics = step(state, actions, gravity, cfg)

assert torch.isfinite(state.position).all()
assert torch.isfinite(state.velocity).all()
assert (state.energy >= 0).all()
print('Device:', device)
print('State position shape:', tuple(state.position.shape))
print('Mean position (m):', state.position.mean(dim=(0,1)).detach().cpu().tolist())
print('Mean remaining energy (J):', state.energy.mean().item())

In [ ]:
#@title 6. Benchmark CPU/GPU physics stepping
# Small and medium populations are useful for comparing launch/transfer overhead.
!python benchmarks/physics_benchmark.py --agents 100 500 1000 2500 5000 --steps 2000 --repeats 5

## P0 interpretation checklist

- [ ] All tests pass.
- [ ] CUDA is detected (optional; CPU execution is valid).
- [ ] Physics outputs remain finite.
- [ ] Energy stays non-negative.
- [ ] CPU and GPU benchmark results are recorded.
- [ ] Runtime, PyTorch version, device, and Git commit are recorded with results.

Do not proceed to controller or evolution implementation until P0 validation passes and the benchmark output has been saved.